# NightWatch - Rough Sleeper Survival Risk Intelligence System
### City of Melbourne Open Data | Data Science Capstone | T1 2026

**Student:** Manya Mahajan (223222623)
**Use Case ID:** UC00210
**Datasets:** Pedestrian Counting System · Microclimate Sensors · Street Furniture · CLUE

## Section 1: Environment Setup & Data Loading

We begin by importing all required libraries and configuring the environment. All datasets are loaded directly from the City of Melbourne Open Data API - no local files are used, ensuring the analysis is fully replicable. A personal API key is used to authenticate requests.

In [ ]:
# Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
from io import StringIO

# Configuration 
warnings.filterwarnings('ignore')
plt.style.use('default')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("NIGHTWATCH — ROUGH SLEEPER SURVIVAL RISK INTELLIGENCE SYSTEM")
print("=" * 70)
print("City of Melbourne Open Data | T1 2026")
print("=" * 70)

## Data Collection Function

We define a reusable function to fetch any dataset from the City of Melbourne Open Data Portal using its dataset ID and a personal API key. The function uses the CSV export endpoint for simplicity and handles errors gracefully.

In [ ]:

API_KEY = "6c7e8c1a6960b266bef6d4294d559502ad85fe8a12795eb6d015fbe3"

def collect_data(dataset_id, api_key, limit=10000):
    """
    Fetch a dataset from the City of Melbourne Open Data API.

    Parameters:
        dataset_id : str — dataset ID from the portal URL
        api_key    : str — personal API key
        limit      : int — max number of records to fetch

    Returns:
        pd.DataFrame or None
    """
    base_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
    url = f"{base_url}{dataset_id}/exports/csv"

    params = {
        "select": "*",
        "limit": limit,
        "lang": "en",
        "timezone": "Australia/Melbourne",
        "apikey": api_key
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            content = response.content.decode("utf-8")
            df = pd.read_csv(StringIO(content), delimiter=";")
            print(f" Loaded '{dataset_id}' successfully: {df.shape[0]:,} rows, {df.shape[1]} columns")
            return df
        else:
            print(f" Failed to load '{dataset_id}'. Status code: {response.status_code}")
            return None
    except Exception as e:
        print(f" Error loading '{dataset_id}': {str(e)}")
        return None

## Data Loading

We load all four NightWatch datasets from the portal. Each dataset serves a distinct role in building the Survival Risk Surface - pedestrian exposure, thermal conditions, shelter availability, and warmth refuge proximity.

In [ ]:
print("\n LOADING DATASETS")
print("-" * 50)

# Correct dataset IDs from data.melbourne.vic.gov.au 
PEDESTRIAN_ID    = "pedestrian-counting-system-monthly-counts-per-hour"
MICROCLIMATE_ID  = "microclimate-sensor-readings"
FURNITURE_ID     = "street-furniture-including-bollards-bicycle-rails-bins-drinking-fountains-horse-"
CLUE_ID          = "business-establishments-per-block-by-clue-industry"

pedestrian       = collect_data(PEDESTRIAN_ID,   API_KEY, limit=50000)
microclimate     = collect_data(MICROCLIMATE_ID,  API_KEY, limit=10000)
street_furniture = collect_data(FURNITURE_ID,     API_KEY, limit=5000)
clue             = collect_data(CLUE_ID,          API_KEY, limit=10000)

# Verify all loaded
datasets = {
    "Pedestrian Counting" : pedestrian,
    "Microclimate Sensors": microclimate,
    "Street Furniture"    : street_furniture,
    "CLUE"                : clue
}

missing = [name for name, df in datasets.items() if df is None]
if missing:
    print(f"\n WARNING: Failed to load: {', '.join(missing)}")
else:
    print("\n All datasets loaded successfully!")

## Section 1: Output Analysis & Observations

All four datasets loaded successfully from the City of Melbourne Open Data API.

| Dataset | Rows | Columns | Note |
| Pedestrian Counting | 50,000 | 9 | Capped at limit - full dataset is much larger |
| Microclimate Sensors | 56 | 9 | Unexpectedly low - likely returning only the most recent readings |
| Street Furniture | 5,000 | 15 | Capped at limit |
| CLUE | 10,000 | 24 | Capped at limit |

**Key observations:**
- The Pedestrian Counting dataset is the largest and will be the primary dataset driving exposure analysis
- Microclimate returning only 56 rows is a concern — this will be investigated in Section 2 and the limit increased or the API query adjusted
- Street Furniture has the richest column count at 15, suggesting good attribute detail for filtering seat and shelter types
- CLUE's 24 columns reflect the breadth of land use categories which will be useful for identifying 24-hour businesses

## Section 2: Initial Data Exploration

We inspect the structure of each dataset-column names, data types, and the first few rows  to understand what variables are available and how they relate to the NightWatch use case.

In [ ]:
for name, df in datasets.items():
    print("=" * 70)
    print(f" {name.upper()}")
    print("=" * 70)
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nData Types:\n{df.dtypes}")
    print(f"\nFirst 2 rows:")
    print(df.head(2))
    print()

## Section 2: Output Analysis & Observations

### Pedestrian Counting
- Contains 9 columns covering sensor ID, date, hour, directional counts, and total pedestrian count
- `sensing_date` is stored as `object`  needs converting to datetime
- `location` contains coordinates as a combined string `"-37.810, 144.964"`  needs splitting into separate latitude and longitude columns for spatial analysis
- `hourday` represents the hour of the day (0–23)  critical for filtering to late-night hours (10pm–5am) for NightWatch

### Microclimate Sensors
- Only 56 rows returned - this is because the dataset returns the **most recent snapshot only** by default, not historical readings
- `type` column stores the reading category (e.g. `TPH.TEMP` for temperature, `WS` for wind speed) - data is in long format, one row per reading type per sensor
- `value` is the actual measurement and `units` confirms the metric (°C, km/h)
- Data shows a 2023 timestamp - the limit needs increasing significantly and a date filter applied to get summer readings
- `site_id` will be used to spatially join with sensor location data

### Street Furniture
- `asset_type` column is the key filter - contains values like `Seat`, `Bollard`, `Bin`, `Planter Box` etc. We only need seats and shelters for NightWatch
- `coordinatelocation` stores coordinates as a combined string - needs splitting like pedestrian dataset
- `condition_rating` is useful - poor condition seats are not usable shelter
- `easting` and `northing` are in MGA Zone 55 projection - will need converting to lat/lon for mapping

### CLUE
- 2024 census data — the most current available, which is ideal
- `food_and_beverage_services` is the key column for NightWatch - these are the warmth refuge candidates (cafes, restaurants, bars open late)
- Data is aggregated at `block_id` level - will need spatially joining to match with pedestrian and microclimate sensor locations
- `clue_small_area` confirms all rows are within Melbourne CBD - consistent with our analysis scope

## Section 3: Data Cleaning & Preprocessing

We address the issues identified in Section 2. The main priorities are:
1. Fix microclimate - increase limit and filter to temperature readings only
2. Convert all date columns to datetime
3. Filter pedestrian data to late-night hours (10pm–5am)
4. Filter street furniture to seats only
5. Split coordinate string columns into separate lat/lon columns

In [ ]:
# ── Try historical microclimate dataset ─────────────────────────
print("Trying historical microclimate dataset...")

base_url = "https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/"
url = f"{base_url}microclimate-sensors-data/exports/csv"

params = {
    "select"  : "*",
    "limit"   : 100000,
    "lang"    : "en",
    "timezone": "Australia/Melbourne",
    "apikey"  : API_KEY
}

response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")

if response.status_code == 200:
    content = response.content.decode("utf-8")
    microclimate = pd.read_csv(StringIO(content), delimiter=";")
    print(f"Shape: {microclimate.shape[0]:,} rows × {microclimate.shape[1]} columns")
    print(f"\nColumns: {list(microclimate.columns)}")
    print(f"\nDate range: {microclimate.iloc[:, 0].min()} to {microclimate.iloc[:, 0].max()}")
    print(f"\nFirst 2 rows:\n{microclimate.head(2)}")
else:
    print(f"Failed with status: {response.status_code}")

## Section 3: Data Cleaning & Preprocessing

With all 4 datasets confirmed, we now clean and filter each one to fit the NightWatch use case scope — overnight hours, temperature readings, seated shelter, and food & beverage businesses.

In [ ]:
# ── 1. PEDESTRIAN — filter to late-night hours (10pm–5am) ───────
pedestrian['sensing_date'] = pd.to_datetime(pedestrian['sensing_date'])
pedestrian_night = pedestrian[pedestrian['hourday'].isin([22, 23, 0, 1, 2, 3, 4, 5])]

print("PEDESTRIAN — Late-night filter")
print(f"  Before: {pedestrian.shape[0]:,} rows")
print(f"  After:  {pedestrian_night.shape[0]:,} rows")
print(f"  Hours kept: {sorted(pedestrian_night['hourday'].unique())}\n")

# ── 2. MICROCLIMATE — convert datetime, drop nulls in key cols ───
microclimate['received_at'] = pd.to_datetime(microclimate['received_at'])
microclimate_clean = microclimate.dropna(subset=['airtemperature', 'latlong'])

print("MICROCLIMATE — Clean temperature readings")
print(f"  Before: {microclimate.shape[0]:,} rows")
print(f"  After:  {microclimate_clean.shape[0]:,} rows\n")

# ── 3. STREET FURNITURE — filter to seats only ───────────────────
print(f"Street furniture asset types available:")
print(f"{street_furniture['asset_type'].value_counts()}\n")

seats = street_furniture[street_furniture['asset_type'] == 'Seat']

print("STREET FURNITURE — Seats only")
print(f"  Before: {street_furniture.shape[0]:,} rows")
print(f"  After:  {seats.shape[0]:,} rows\n")

# ── 4. CLUE — filter to food & beverage only ────────────────────
clue_clean = clue[clue['food_and_beverage_services'] > 0].copy()

print("CLUE — Blocks with food & beverage businesses")
print(f"  Before: {clue.shape[0]:,} rows")
print(f"  After:  {clue_clean.shape[0]:,} rows")

## Section 3: Output Analysis & Observations

### Pedestrian Counting
- Filtering to late-night hours (10pm–5am) reduced rows from 50,000 to 15,125 - meaning roughly 30% of all pedestrian activity in this dataset occurs overnight, which is significant for a use case focused on rough sleeper exposure

### Microclimate
- 4,124 rows dropped due to null temperature or location values - likely sensor outages or connectivity issues, which is normal for IoT sensor networks
- 95,876 clean temperature readings remain - more than sufficient for analysis

### Street Furniture
- Seats represent only 701 of 5,000 records (14%) - the majority of street furniture is bollards and bicycle rails which are not relevant to NightWatch
- 701 seat locations across the CBD will be mapped as potential shelter points for rough sleepers

### CLUE
- 4,763 blocks contain at least one food & beverage establishment - these are the warmth refuge candidates (cafes, bars, restaurants) that rough sleepers may access overnight
- Blocks with zero food & beverage presence have been excluded as they offer no warmth refuge potential